# Replaying experimental recordings and inferring dynamical quantities

In this tutorial, we replay a snippet of experimentally recorded fly walking by matching recorded joint angle sequences with position actuators. In other words, the simulation will compare current joint angles with target (i.e., recorded) joint angles, and apply a force to correct for the difference.

We start by loading a snippet of experimentally recorded kinematics. A `MotionSnippet` helper class is implemented for this purpose.

In [ ]:
import numpy as np
from flygym_examples.spotlight_data import MotionSnippet
 
snippet = MotionSnippet()

This recording was collected using the [Spotlight](https://go.epfl.ch/spotlight-poseforge) system, and the 3D pose was inferred using PoseForge. This workflow is described in [Wang-Chen et al., 2026](https://www.biorxiv.org/content/10.64898/2026.03.11.711180).

Let's inspect what this dataset contains:

In [ ]:
print("Experimental data and metadata:")
print(f"  legs: {snippet.legs}")
print(f"  dofs_per_leg: {snippet.dofs_per_leg}")
print(f"  data_fps: {snippet.data_fps}")
print(f"  experiment_trial: {snippet.experiment_trial}")
print(f"  framerange_in_raw_recording: {snippet.framerange_in_raw_recording}")
print(f"  joint_angles: shape={snippet.joint_angles.shape}, dtype={snippet.joint_angles.dtype}")
print(f"  fwdkin_egoxyz: shape={snippet.fwdkin_egoxyz.shape}, dtype={snippet.fwdkin_egoxyz.dtype}")
print(f"  rawpred_egoxyz: shape={snippet.rawpred_egoxyz.shape}, dtype={snippet.rawpred_egoxyz.dtype}")

As a sanity check, we can plot the egocentric _x_ (fore/aft), _y_ (left/right), and _z_ (height) positions of the claw of the left front leg over time.

It is worth noting that the dataset contains two versions of the _xyz_ keypoint positions:

1. The raw coordinates predicted by the PoseForge model (`rawpred_egoxyz`). These are outputs of a black-box deep learning model and do not _explicitly_ take anatomical contraints into account. We will plot these iasn black dotted lines.
2. The anatomically constrained coordinates following inverse and forward kinematics (`fwdkin_egoxyz`). That is, joint angles in a purely geometic model (no physics simulation) are fitted to minimize the discrepancy between the geometric model's keypoint positions and the raw model outputs. As such, these fitted keypoint positions are constrained by (a) leg segment lengths and (b) the available joint rotation axes (e.g., only two axes of rotation are possible at the femur-tibia joint, and only one is available at the tibia-tarsus joint). The process of fitting joint angles is called _[inverse kinematics](https://en.wikipedia.org/wiki/Inverse_kinematics)_, and the process of computing the resulting constrained _xyz_ coordinates is called _[forward kinematics](https://en.wikipedia.org/wiki/Forward_kinematics)_. This dataset uses [SeqIKPy](https://nely-epfl.github.io/sequential-inverse-kinematics/) ([Özdil et al., 2026](https://doi.org/10.21105/joss.08557)) for inverse and forward kinematics, but many other options are available (e.g., [DeepLabCut](https://doi.org/10.1038/s41593-018-0209-y) + [Anipose](https://doi.org/10.1016/j.celrep.2021.109730), and [DeepFly3D](https://elifesciences.org/articles/48571)). We will plot these in blue.

In [ ]:
import matplotlib.pyplot as plt

leg_to_plot = "lf"
leg_to_plot_idx = snippet.legs.index(leg_to_plot)
n_kpts_per_leg = 5
kpt_idx_to_visualize = leg_to_plot_idx * n_kpts_per_leg + n_kpts_per_leg - 1
expdata_timegrid = np.arange(snippet.joint_angles.shape[0]) / snippet.data_fps

fig, axes = plt.subplots(3, 1, figsize=(8, 3.5), tight_layout=True, sharex=True)
for dim_idx, dim_name in enumerate("xyz"):
    ax = axes[dim_idx]
    ts_fwdkin = snippet.fwdkin_egoxyz[:, kpt_idx_to_visualize, dim_idx]
    ts_rawpred = snippet.rawpred_egoxyz[:, kpt_idx_to_visualize, dim_idx]
    ax.plot(
        expdata_timegrid,
        ts_fwdkin,
        label="Forward kinematics",
        color="C0",
    )
    ax.plot(
        expdata_timegrid,
        ts_rawpred,
        label="Raw prediction",
        linestyle=":",
        color="black",
    )

    # ignorable plotting code below
    _center = np.percentile(ts_rawpred, [10, 90]).mean()
    _plotspan = 2  # mm
    ax.set_ylim(_center - 0.5 * _plotspan, _center + 0.5 * _plotspan)
    ax.set_ylabel(f"Distance\n(mm)")
    ax.set_title(dim_name, fontsize="medium")
    if dim_name == "z":
        ax.set_xlabel("Time (s)")
    if dim_name == "x":
        ax.legend(bbox_to_anchor=(1.04, 0.5), loc="center left")

fig.suptitle(f"{leg_to_plot.upper()} leg claw position (egocentric Cartesian frame)")

We can also plot all legs in 3D, but only for one frame:

In [ ]:
frame_to_plot = 0
 
ax = plt.figure(figsize=(5, 3), tight_layout=True).add_subplot(projection="3d")
for leg_idx, leg_name in enumerate(snippet.legs):
    kpt_indices = slice(leg_idx * n_kpts_per_leg, (leg_idx + 1) * n_kpts_per_leg)
    fwdkin_egoxyz = snippet.fwdkin_egoxyz[frame_to_plot, kpt_indices, :]
    rawpred_egoxyz = snippet.rawpred_egoxyz[frame_to_plot, kpt_indices, :]
    ax.plot(
        fwdkin_egoxyz[:, 0],
        fwdkin_egoxyz[:, 1],
        fwdkin_egoxyz[:, 2],
        label=f"{leg_name.upper()} leg",
        color=f"C{leg_idx}",
    )
    ax.plot(
        rawpred_egoxyz[:, 0],
        rawpred_egoxyz[:, 1],
        rawpred_egoxyz[:, 2],
        label="Raw prediction" if leg_idx == 5 else None,
        linestyle=":",
        color="black",
    )
ax.set_xlabel("x (mm)")
ax.set_ylabel("y (mm)")
ax.set_zlabel("z (mm)")
ax.set_aspect("equal")
ax.legend(loc="center left", bbox_to_anchor=(1.3, 0.5))

Now, let's visualize the joint angles time series fitted through inverse kinematics:

In [ ]:
fig, axes = plt.subplots(7, 1, figsize=(7, 6.5), tight_layout=True, sharex=True)
for dof_idx, (parent_link, child_link, axis) in enumerate(snippet.dofs_per_leg):
    ax = axes[dof_idx]
    ts_angles = np.rad2deg(snippet.joint_angles[:, leg_to_plot_idx, dof_idx])
    ax.plot(expdata_timegrid, ts_angles)
 
    _center = np.percentile(ts_angles, [10, 90]).mean()
    _plotspan = 120  # deg
    ax.set_ylim(_center - 0.5 * _plotspan, _center + 0.5 * _plotspan)
    ax.set_ylabel("Angle\n(°)")
    ax.set_title(f"{parent_link}-{child_link} {axis}", fontsize="medium")
    if dof_idx == 6:
        ax.set_xlabel("Time (s)")
fig.suptitle(f"{leg_to_plot.upper()} leg joint angles")

We are now ready to instantiate a fly model, attach it to a world, and create a simulation object. Detailed explanations are available in _Tutorial 1: Composing models and scenes_.

In [ ]:
from flygym.compose import Fly, KinematicPosePreset, ActuatorType
from flygym.anatomy import Skeleton, AxisOrder, JointPreset, ActuatedDOFPreset

axis_order = AxisOrder.YAW_PITCH_ROLL
articulated_joints = JointPreset.ALL_BIOLOGICAL
actuated_dofs = ActuatedDOFPreset.LEGS_ACTIVE_ONLY
neutral_pose = KinematicPosePreset.NEUTRAL
actuator_type = ActuatorType.POSITION

# This controlls how strongly the actuators try to track the target joint angles
actuator_gain = 150.0  # in uN*mm/rad (torque applied per angular discrepancy)
 
fly = Fly()
skeleton = Skeleton(axis_order=axis_order, joint_preset=articulated_joints)
fly.add_joints(skeleton, neutral_pose=neutral_pose)
 
actuated_dofs_list = fly.skeleton.get_actuated_dofs_from_preset(actuated_dofs)
fly.add_actuators(
    actuated_dofs_list,
    actuator_type=actuator_type,
    kp=actuator_gain,
    neutral_input=neutral_pose,
)
 
fly.colorize()
tracking_cam = fly.add_tracking_camera()

The only thing worth noting here that was not mentioned in the previous tutorials is that insects have specialized structures on thir legs that allow them to adhere to surfaces. We can use `fly.add_leg_adhesion` to emulate this. However, note that adding adhesion in the model does not do anything by itself—the adhesion actuators must be switched on (or off) during simulation time, as demonstrated in a later code block.

In [ ]:
fly.add_leg_adhesion()

Now we create a world, attach the fly to it, set up the simulation, and add a renderer:

In [ ]:
from flygym.anatomy import JointDOF, RotationAxis, BodySegment
from flygym.compose import FlatGroundWorld
from flygym.utils.math import Rotation3D
from flygym import Simulation

spawn_pos = [0, 0, 0.7]  # center of thorax is at 0.7 mm above the ground
spawn_rot = Rotation3D(format="quat", values=[1, 0, 0, 0])  # no rotation
 
world = FlatGroundWorld()
world.add_fly(fly, spawn_pos, spawn_rot)
 
sim = Simulation(world)
sim.set_renderer(tracking_cam)

Here we call `MotionSnippet.get_joint_angles` to produce the actuator target sequence. Internally this does three things:

1. The Spotlight system records data at 330 Hz, which is much slower than NeuroMechFly simulation (typically 10 kHz). The recorded joint angles are therefore upsampled to the target frequency. More precisely, a 3rd-order Savitzky-Golay filter with a very small time window (0.03 s) is first applied to the time series; the finer time series is then fitted using cubic-spline interpolation.
2. The data is reshaped into a NumPy array of shape (n_steps, n_actuated_dofs), with the DoFs reordered to match the order expected by FlyGym.
3. FlyGym (after v2.0.0) defines joint angles anatomically—that is, "outward" rotations have the same sign on both left- and right-side legs even though they have opposite directions of rotation relative to the world frame. This is not the case in SeqIKPy, so this function flips the sign for roll and yaw DoFs on the right-side legs (pitch is not affected).

In [ ]:
sim_timestep = 1e-4
 
joint_angles_nmf = snippet.get_joint_angles(
    output_timestep=sim_timestep,
    output_dof_order=fly.get_actuated_jointdofs_order(actuator_type),
)
 
sim_duration_sec = snippet.joint_angles.shape[0] / snippet.data_fps
nmfsim_timegrid = np.arange(0, sim_duration_sec, sim_timestep)
 
print("Number of steps to simulate:", nmfsim_timegrid.size)
print("Shape of target joint angles:", joint_angles_nmf.shape)

Now we implement the main simulation loop. At each timestep we feed the precomputed target angles to the position actuators, step the physics forward, and record the resulting joint angles and actuator torques. We also call the renderer at each step; the FlyGym renderer internally determines whether a frame should actually be rendered based on the user-specified output playback speed and output frame rate (in this case we use the default parameters when we called `sim.set_renderer`).

Note that leg adhesion is turned on before the loop and kept on throughout.

In [ ]:
from tqdm import trange

fly_name = fly.name
nsteps_sim = nmfsim_timegrid.size
n_dofs = len(fly.get_jointdofs_order())
n_actuated_dofs = len(fly.get_actuated_jointdofs_order(actuator_type))
 
simulated_joint_angles = np.full((nsteps_sim, n_dofs), np.nan, dtype=np.float32)
actuator_torques = np.full((nsteps_sim, n_actuated_dofs), np.nan, dtype=np.float32)
 
sim.reset()
# Turn adhesion on for all 6 legs
sim.set_leg_adhesion_states(fly_name, np.ones(6, dtype=np.bool))
# sim.warmup()
 
for step_idx in trange(nsteps_sim, desc="Simulating"):
    # Set actuator inputs
    target_angles = joint_angles_nmf[step_idx, :]
    sim.set_actuator_inputs(fly_name, actuator_type, target_angles)
    
    # Step simulation
    sim.step_with_profile()  # can be replaced with sim.step()
    
    # Record simulation data
    simulated_joint_angles[step_idx, :] = sim.get_joint_angles(fly_name)
    actuator_torques[step_idx, :] = sim.get_actuator_forces(fly_name, actuator_type)
    
    # Render as needed (flygym internally decides whether to actually render a frame
    # based on renderer configs)
    sim.render_as_needed_with_profile()  # can be replaced with sim.render_as_needed()

Observe that we have called `sim.step_with_profile()` and `sim.render_as_needed_with_profile` to time code execution behind the scenes. This allows us to use `sim.print_performance_report()` to see a breakdown of compute time shown below. In practice, you can replace these with `sim.step()` and `sim.render_as_needed()`, which ironically should be a bit faster because we remove the timing overhead.

In [ ]:
sim.print_performance_report()

Let's play the rendered video in the notebook:

In [ ]:
sim.renderer.show_in_notebook()

Finally, we compare the actuator target angles (blue) against the angles actually achieved by the simulator (orange):

In [ ]:
fig, axes = plt.subplots(7, 1, figsize=(9, 6.5), tight_layout=True, sharex=True)
for dof_idx, (parent_link, child_link, axis) in enumerate(snippet.dofs_per_leg):
    ax = axes[dof_idx]

    if parent_link == "thorax":
        parent_name = "c_thorax"
    else:
        parent_name = f"{leg_to_plot}_{parent_link}"
    child_name = f"{leg_to_plot}_{child_link}"
    target_dof = JointDOF(BodySegment(parent_name), BodySegment(child_name), RotationAxis(axis))

    # Index into joint_angles_nmf (actuated DOF order)
    actuated_dof_index = fly.get_actuated_jointdofs_order(actuator_type).index(target_dof)
    ts_target = np.rad2deg(joint_angles_nmf[:, actuated_dof_index])
    ax.plot(nmfsim_timegrid, ts_target, label="Target", linestyle=":", color="C0")

    # Index into simulated_joint_angles (all-DOF order)
    nmf_dof_index = fly.get_jointdofs_order().index(target_dof)
    ts_sim = np.rad2deg(simulated_joint_angles[:, nmf_dof_index])
    ax.plot(nmfsim_timegrid, ts_sim, label="Simulated", color="C1")

    _center = np.percentile(ts_target, [10, 90]).mean()
    _plotspan = 120  # deg
    ax.set_ylabel("Angle\n(°)")
    ax.set_title(f"{parent_link}-{child_link} {axis}", fontsize="medium")
    if dof_idx == 6:
        ax.set_xlabel("Time (s)")
    if dof_idx == 0:
        ax.legend(bbox_to_anchor=(1.04, 0.5), loc="center left")